In [1]:
import torch
import os
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
import huggingface_hub

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt 

from timm import create_model  # or torchvision.models
from tqdm.notebook import tqdm
from torchinfo import summary

from torch.utils.data import DataLoader, random_split

In [2]:
import sys, os

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    CODE_DIR = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code'
else:
    CODE_DIR = os.path.abspath('../project/src')

sys.path.insert(0, CODE_DIR)
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))
sys.path.insert(0, os.path.abspath('../project') if not IS_KAGGLE else '/kaggle/input/hms-eeg-code')

import yaml
CONFIG_PATH = os.path.join(CODE_DIR, 'config.yaml') if IS_KAGGLE else '../project/config/config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
print(cfg)

{'seed': 42, 'deterministic': True, 'data': {'train_meta': 'data_meta_splits/train_test_split.csv', 'val_meta': 'data_meta_splits/train_test_split.csv', 'eeg_dir': 'data_eeg_preprocessed/eeg_npy', 'spec_dir': 'data_eeg_preprocessed/spec_npy'}, 'model': {'name': 'eeg_1d_small', 'num_classes': 6}, 'loss': {'name': 'ce', 'class_weight': [], 'label_smoothing': 0.0}, 'train': {'run_name': 'exp', 'out_dir': 'runs', 'batch_size': 64, 'num_workers': 8, 'epochs': 50, 'lr': 0.0015, 'weight_decay': 0.01, 'amp': True, 'grad_clip': 1.0, 'use_weighted_sampler': False, 'early_stop_patience': 10, 'early_stop_key': 'macro_f1', 'lr_mode': 'cos'}, 'log': {'use_wandb': False, 'project': 'EEG-HBA'}}


In [3]:
# ========== ENVIRONMENT CONFIG ==========
if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    META_PATH = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
else:
    DATA_ROOT = os.path.abspath('../')
    META_PATH = os.path.abspath('../data_meta_splits/train_test_split.csv')

# ========== REPRODUCIBILITY ==========
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | Device: {DEVICE}")

Environment: Kaggle | Device: cuda


In [4]:
from data import EEGDataset
from torch.utils.data import DataLoader

PARQUET_DIR = os.path.join(DATA_ROOT, 'train_eegs') if IS_KAGGLE else None
NPY_DIR = None if IS_KAGGLE else os.path.join(DATA_ROOT, cfg["data"]["eeg_dir"])

train_ds = EEGDataset(META_PATH, npy_dir=NPY_DIR, split="train", parquet_dir=PARQUET_DIR)
val_ds   = EEGDataset(META_PATH, npy_dir=NPY_DIR, split="val",   parquet_dir=PARQUET_DIR)

train_loader = DataLoader(train_ds, batch_size=cfg["train"]["batch_size"],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg["train"]["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Train: 66251 | Val: 17179


In [5]:
# Batch shape verification
batch = next(iter(train_loader))
print("x shape:", batch["x"].shape)
print("y shape:", batch["y"].shape)
print("y sample:", batch["y"][:4])

x shape: torch.Size([64, 20, 10000])
y shape: torch.Size([64])
y sample: tensor([4, 2, 4, 1])


In [6]:
from models.classifier import build_model

model = build_model(cfg["model"]).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total_params:,} | trainable: {trainable:,}")

Parameters — total: 217,030 | trainable: 217,030


In [ ]:
# v2: soft label + KL div loss
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score
import time

NUM_EPOCHS  = cfg["train"]["epochs"]
LR          = cfg["train"]["lr"]
GRAD_CLIP   = cfg["train"]["grad_clip"]
USE_AMP     = cfg["train"]["amp"] and DEVICE.type == "cuda"

criterion     = nn.KLDivLoss(reduction='batchmean')
val_criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR,
                  weight_decay=cfg["train"]["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_f1, history = 0.0, []
patience = cfg["train"].get("early_stop_patience", 10)
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- train ---
    model.train()
    t0 = time.time()
    train_loss = 0.0
    for batch in train_loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(x)
            log_probs = F.log_softmax(logits, dim=1)
            loss = criterion(log_probs, batch["soft_y"].to(DEVICE))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    # --- validate ---
    model.eval()
    val_loss, val_kl, all_preds, all_labels = 0.0, 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(x)
            val_loss += val_criterion(logits, y).item()
            val_kl   += criterion(F.log_softmax(logits, dim=1), batch["soft_y"].to(DEVICE)).item()
            all_preds .extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    macro_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    avg_train  = train_loss / len(train_loader)
    avg_val    = val_loss   / len(val_loader)
    avg_val_kl = val_kl     / len(val_loader)
    elapsed    = time.time() - t0

    history.append({"epoch": epoch, "train_kl": avg_train,
                    "val_kl": avg_val_kl, "val_ce": avg_val, "macro_f1": macro_f1})

    print(f"Epoch {epoch:03d} | train_kl {avg_train:.4f} | "
          f"val_kl {avg_val_kl:.4f} | val_ce {avg_val:.4f} | macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        wait = 0                 # ← 加这行
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ✓ saved best model (f1={best_f1:.4f})")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best macro_f1: {best_f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs    = [h["epoch"]    for h in history]
train_kl  = [h["train_kl"] for h in history]
val_kl    = [h["val_kl"]   for h in history]
val_ce    = [h["val_ce"]   for h in history]
macro_f1  = [h["macro_f1"] for h in history]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

ax1.plot(epochs, train_kl, label="train KL")
ax1.plot(epochs, val_kl,   label="val KL")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("KL Divergence")
ax1.set_title("KL Divergence (train vs val)"); ax1.legend()

ax2.plot(epochs, val_ce, color="orange", label="val CE")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Cross-Entropy Loss")
ax2.set_title("Val CE Loss (monitoring)"); ax2.legend()

ax3.plot(epochs, macro_f1, color="green")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Macro F1")
ax3.set_title("Validation Macro F1")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

print(f"\nFinal epoch summary:")
print(f"  Best macro_f1 : {max(macro_f1):.4f}  (epoch {epochs[macro_f1.index(max(macro_f1))]})")
print(f"  Best val KL   : {min(val_kl):.4f}  (epoch {epochs[val_kl.index(min(val_kl))]})")
print(f"  Final train KL: {train_kl[-1]:.4f}")
print(f"  Final val KL  : {val_kl[-1]:.4f}")